# OpenCoder: four independent benchmark arms

Official model: [infly/OpenCoder-8B-Instruct](https://huggingface.co/infly/OpenCoder-8B-Instruct).
Pinned revision: `cc9efae8dc513fe21e4b2572ea6ccacd4e68998b` (native Llama architecture and official chat template).

| Arm | Training data | Epochs |
|---|---|---|
| base | None | 0 |
| qa | Original QA for the matched 1,717 admitted sources | 1 |
| mcq_original | 1,717 originals with approved corrections | 1 |
| mcq_augmented | 5,151 originals + two variants per source | 1 |

Every trained arm starts from the original OpenCoder weights with fresh LoRA/optimizer.
There is no sequential training between arms and no Qwen adapter loading.
All MCQs use one deterministic shuffled option order; training examples shuffle.
Validation (192) and test (269) are unchanged, with identical MCQ prompt/parser,
greedy decoding and 128 generated tokens for all four arms. Test is scored only
after each arm finishes; smoke never reads test answers for scoring.

NF4 double quantization, fp16, batch1/accumulation16, LR1e-4, cosine/5% warmup,
LoRA rank16/alpha32/dropout.05, weight decay.01, clip.3, answer-only loss, seed42.
Context cap8,192; no truncation or dropped groups. Preflight prints actual lengths.
Each trained arm saves three within-epoch checkpoints with validation loss and accuracy.
QA also uses MCQ validation targets, so its validation loss measures MCQ answering.
The longer augmented epoch has more optimizer steps: this is not a matched-budget ablation.
Variants are accepted as supplied under your no-review instruction.

Import, attach the private dataset, enable GPU and Internet, then Save & Run All.
GPU smoke checks longest presentations separately for all three trained arms.
Smoke adapters are discarded; full runs initialize independently.

Tokenizer recovery: explicitly trust the official custom tokenizer at the pinned model commit. Uses the existing attached dataset.


In [ ]:
import os
os.environ['CUDA_VISIBLE_DEVICES'] = '0'
os.environ['USE_TF'] = '0'
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
%pip install -q 'transformers==4.57.3' 'peft==0.17.1' 'accelerate==1.15.0' 'bitsandbytes==0.48.2' sentencepiece matplotlib


In [ ]:
import csv, hashlib, json, shutil, subprocess, sys, zipfile
from pathlib import Path
from datetime import datetime, timezone
WORK = Path('/kaggle/working')
os.chdir(WORK)
EXPECTED = {'acceptance_policy.json': 'df0c1d9f7b16e98cd135b4616e42dfadd07010846378d39e686b5f319c3c12ac', 'approved_five_fixes_receipt.json': 'baaed911806bdd26ea341bbf852d7390db979617162c46993768c1341fae6fc2', 'augmentation_manifest.json': 'fdfc5970b0ebf470c49a1b66dab91a755bb115f1766eb1f5691bb4a18109b8c3', 'benchmark.py': '03173706c121f1ddc136cce1573732861c46a714cc5905292bc6a2cd055915c6', 'experiment.json': '8516685ff790c9ecc693ab1a033310138fb69c4a0a6e478dac1cc8c3f30b4c9f', 'finetune_qwen_coder.py': '434e2eb3fb1e29e2e9bc16d4564d1499175000c00cc6daf3b7a03115ec6a23a9', 'finetune_qwen_mcq.py': 'a498843abf38a15a13d1c6604da346d767334cab90d55a1db65fe7b96787da67', 'mcq_augmented.csv': '8d07ab7847bdc1e4f85f363122813a7d1a91d4ec850776bf552119bb80aacb20', 'mcq_checkpointing.py': '5f9672eaa9943ec1333de5fe1112eb7c652164cb0a8c49091e7b51a452421379', 'mcq_original.csv': 'a4d2ce3fa8310d66fab0924693f6b7c2829089c5f4ead4081bccf3f063d00767', 'opencoder_benchmark.py': 'acddf0b1bc169fdbd2d381be99e1e9d0cf93eb369a9bf6fba9adbdc67d1b9c9f', 'original_repairs.json': 'a795c29de7492e7eed5e3fbc93cd681ef873e3c29e2f02203ef77cb1128c261c', 'qa.csv': '79297cfa9bed406cd6138e38b3152e113081496c4b19baa60ea18952dbad9b73', 'TestData.csv': '5e10ec19d61624ee3554814337cf961fe499f4a666881c866c172fa1e51454a2', 'tests/__init__.py': 'dcd7f82a00ec26a2767168bbf833e21705e158c6fcbd2fcae799fc43836d768a', 'tests/test_opencoder_benchmark.py': 'a86ea956f82c1a1900397e5a8eaa744041e1e3e529a3f4450302c008308dc675', 'validation.csv': 'c7c028971d4ba185113eefafbb74260db4e5484542c04fec96334e1b18c77e20'}
matches = [p.parent for p in Path('/kaggle/input').rglob('experiment.json')
           if all((p.parent / name).is_file() for name in EXPECTED)]
assert len(matches) == 1, f'Attach exactly one OpenCoder benchmark dataset: {matches}'
DATA = matches[0]
for name, digest in EXPECTED.items():
    assert hashlib.sha256((DATA / name).read_bytes()).hexdigest() == digest, name
    target = WORK / name
    target.parent.mkdir(parents=True, exist_ok=True)
    shutil.copyfile(DATA / name, target)
# OpenCoder tokenizer hotfix: applied after verifying/copying existing dataset inputs.
p = WORK / 'opencoder_benchmark.py'
source = p.read_text(encoding='utf-8')
old = 'AutoTokenizer.from_pretrained(MODEL_ID, revision=REVISION)'
new = 'AutoTokenizer.from_pretrained(MODEL_ID, revision=REVISION, code_revision=REVISION, trust_remote_code=True, use_fast=False)'
assert old in source or new in source, 'Unexpected benchmark script; use the updated bundle'
p.write_text(source.replace(old, new), encoding='utf-8')

RESULT = WORK / ('opencoder-four-arm-' + datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ'))
RESULT.mkdir()
for name in EXPECTED:
    if name.endswith('.py') or name in ['experiment.json', 'augmentation_manifest.json', 'acceptance_policy.json', 'original_repairs.json']:
        target = RESULT / 'provenance' / name
        target.parent.mkdir(parents=True, exist_ok=True)
        shutil.copyfile(WORK / name, target)
(RESULT / 'input_checksums.json').write_text(json.dumps(EXPECTED, indent=2))
# Keep defaults for the full four-arm comparison. A subset can run in another version.
ARMS = ['base', 'qa', 'mcq_original', 'mcq_augmented']
MAX_LENGTH = 8192
ROTATED_VALIDATION = True
RUN_SMOKE = True
def archive():
    return shutil.make_archive(str(RESULT), 'zip', RESULT)
def run(name, arm='base', *flags):
    command = [sys.executable, '-u', 'opencoder_benchmark.py', '--data', str(WORK),
        '--output', str(RESULT / name), '--arm', arm, '--max-length', str(MAX_LENGTH), *flags]
    try:
        with (RESULT / (name + '.log')).open('w', encoding='utf-8') as log:
            process = subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                text=True, bufsize=1, env={**os.environ, 'PYTHONUNBUFFERED': '1'})
            for line in process.stdout:
                print(line, end='', flush=True)
                log.write(line); log.flush()
            if process.wait():
                raise RuntimeError(f'{name} failed; partial outputs archived')
    finally:
        archive()
subprocess.run([sys.executable, '-m', 'unittest', 'tests.test_opencoder_benchmark', '-v'], check=True)
run('preflight', 'base', '--prepare-only')
print(json.loads((RESULT / 'preflight/metadata.json').read_text())['budgets'])


In [ ]:
if RUN_SMOKE:
    for arm in ARMS:
        if arm != 'base':
            run('smoke-' + arm, arm, '--smoke')
            report = json.loads((RESULT / ('smoke-' + arm) / 'gradient_check.json').read_text())
            assert report['finite_nonzero_lora_gradients'] and report['base_frozen']
            assert json.loads((RESULT / ('smoke-' + arm) / 'metadata.json').read_text())['saved_adapter_reload_verified']
print('Smoke stage complete. Full arms start from the original OpenCoder weights.')


In [ ]:
for arm in ARMS:
    flags = ['--rotated-validation'] if ROTATED_VALIDATION else []
    run(arm, arm, *flags)
    if arm != 'base':
        scores = json.loads((RESULT / arm / 'checkpoint_scores.json').read_text())
        assert len(scores) == 3
        print(arm, 'checkpoint validation scores:', scores)


In [ ]:
import matplotlib.pyplot as plt
comparison = []
for arm in ARMS:
    summary = json.loads((RESULT / arm / 'summary.json').read_text())
    metadata = json.loads((RESULT / arm / 'metadata.json').read_text())
    comparison.append({'arm': arm, 'validation_accuracy': summary['validation']['accuracy'],
        'test_accuracy': summary['test']['accuracy'], 'test_correct': summary['test']['correct'],
        'test_invalid': summary['test']['invalid_answers'],
        'rotated_validation_accuracy': summary.get('validation_rotated', {}).get('accuracy'),
        'training_presentations': 0 if arm == 'base' else metadata['budgets'][arm]['presentations'],
        'optimizer_steps': 0 if arm == 'base' else metadata['budgets'][arm]['optimizer_steps'],
        'seconds': metadata['elapsed_seconds']})
    log_path = RESULT / arm / 'training_log.json'
    if log_path.exists():
        logs = [r for r in json.loads(log_path.read_text()) if 'loss' in r]
        plt.plot([r['step'] for r in logs], [r['loss'] for r in logs], label=arm)
plt.xlabel('Optimizer step'); plt.ylabel('Training loss (QA and MCQ targets differ)')
plt.legend(); plt.savefig(RESULT / 'loss.png', bbox_inches='tight'); plt.show()
with (RESULT / 'comparison.csv').open('w', newline='') as handle:
    writer = csv.DictWriter(handle, fieldnames=list(comparison[0]))
    writer.writeheader(); writer.writerows(comparison)
(RESULT / 'comparison.json').write_text(json.dumps(comparison, indent=2))
paired = {}
if 'base' in ARMS:
    def predictions(arm):
        with (RESULT / arm / 'test/predictions.csv').open(newline='') as handle:
            return list(csv.DictReader(handle))
    base = predictions('base')
    for arm in ARMS:
        if arm == 'base': continue
        tuned = predictions(arm)
        assert len(base) == len(tuned) == 269
        assert all(a['question'] == b['question'] and a['truth'] == b['truth'] for a,b in zip(base,tuned))
        paired[arm] = {'wrong_to_correct': sum(a['correct']=='False' and b['correct']=='True' for a,b in zip(base,tuned)),
            'correct_to_wrong': sum(a['correct']=='True' and b['correct']=='False' for a,b in zip(base,tuned))}
(RESULT / 'paired_changes.json').write_text(json.dumps(paired, indent=2))
print(json.dumps(comparison, indent=2)); print('Paired changes:', paired)
from IPython.display import display, FileLink
display(FileLink(archive()))
